#Overall architecture

from IPython.display import display, HTML


<pre style="
  background:#1e1e2e;
  color:#cdd6f4;
  padding:24px;
  border-radius:12px;
  font-size:14px;
  line-height:2;
  font-family:monospace;
">
👤 Widget: User Input (Market / Time Range)
              │
              ▼
🎯 Python: Orchestrator — prepare data context
              │
              ▼
📊 Agent: Overview — generate headline metrics
              │
              ▼
🔍 Agent: Anomaly Detector — check for irregularity
         │                    │
   Has anomaly           No anomaly
         │                    │
         ▼                    │
🔬 Agent: Deep Dive           │
   break down drivers         │
         │                    │
         └──────────┬─────────┘
                    ▼
📝 Agent: Summary — executive summary + recommendations
</pre>

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import date, timedelta
import json
# Save outputs
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/Gen-AI/sales-agent/data'
os.makedirs(save_dir, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import google.generativeai as genai
from google.colab import userdata

# API setup
api_key = userdata.get('API_Agent')
genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash-lite")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


#Load Data and Schema

In [3]:
df_revenue = pd.read_csv(f"{save_dir}/monthly_revenue.csv")
df_customer = pd.read_csv(f"{save_dir}/customers.csv")
df_churn = pd.read_csv(f"{save_dir}/monthly_churn.csv")

with open(f"{save_dir}/schema.json", "r") as file:
    schema = file.read()

# Custom queries (Data analysts write own queries)

In [4]:
start_date = '2022-01-01'
end_date = '2024-12-31'

##Query library and function

In [5]:
import duckdb

# Register dataframes as SQL tables
con = duckdb.connect()
con.register('customers', df_customer)
con.register('monthly_revenue', df_revenue)
con.register('monthly_churn', df_churn)

def run_query(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

In [6]:
df_revenue.head()

,customer_id,month,market,segment,plan,product,mrr,event_type
0,SLEEK-0001,2023-04,HK,Startup,Starter,Incorporation,790.65,new
1,SLEEK-0001,2023-04,HK,Startup,Starter,Accounting,42.64,new
2,SLEEK-0001,2023-04,HK,Startup,Starter,Compliance,28.66,new
3,SLEEK-0001,2023-05,HK,Startup,Starter,Accounting,40.95,active
4,SLEEK-0001,2023-05,HK,Startup,Starter,Compliance,28.02,active


In [7]:
df_customer.head()

,customer_id,signup_date,market,segment,plan,acquisition_channel,is_foreign_founder,headcount,cac,churned,churn_date,ltv
0,SLEEK-0001,2023-04-03,HK,Startup,Starter,Organic,False,6,75.69,False,NaN,2258.43
1,SLEEK-0002,2022-01-18,SG,Startup,Starter,Organic,False,3,84.80,False,NaN,3356.15
2,SLEEK-0003,2023-02-11,SG,Startup,Starter,Partner,False,2,132.92,True,2024-01,1537.63
3,SLEEK-0004,2023-12-15,SG,Startup,Starter,Partner,False,6,178.39,False,NaN,1652.99
4,SLEEK-0005,2022-01-02,SG,SME,Growth,Partner,False,22,166.38,True,2023-03,11979.12


#Orchastrator

In [8]:
def orchastrator(market= ['All'], start_date=start_date, end_date=end_date) -> dict:

  #market filter
  if market == ['All']:

    market_list = tuple(df_revenue['market'].unique())

  else:
    market_list = tuple(market)

  mrr_trend=f"""

  select
    month
    ,market
    ,segment
    ,plan
    ,product
    ,event_type
    ,sum(mrr) as monthly_revenue
  from monthly_revenue
  where
    product != 'Incorporation'
    and month between substring('{start_date}',1,7) and substring('{end_date}',1,7)
    and market in {market_list}
  group by month, market, segment, plan, product, event_type

  """

  churn_query = f"""

  select
    churn_month as month
    ,market
    ,segment
    ,plan
    ,churn_reason_category
    ,sum(mrr_lost) as monthly_mrr_lost
    ,count(customer_id) as customers_lost
  from monthly_churn
  where
    churn_month between substring('{start_date}',1,7) and substring('{end_date}',1,7)
    and market in {market_list}
  group by churn_month, market, segment, plan, churn_reason_category

  """

  cac_ltv_query = f"""
  SELECT
      acquisition_channel,
      market,
      segment,
      AVG(cac) as avg_cac,
      AVG(ltv) as avg_ltv,
      AVG(ltv / cac) as ltv_cac_ratio,
      COUNT(customer_id) as customers
  FROM customers
  WHERE market IN {market_list}
  AND signup_date BETWEEN '{start_date}' AND '{end_date}'
  GROUP BY acquisition_channel, market, segment
  ORDER BY avg_cac DESC
  """

  df_mrr_trend = run_query(mrr_trend)
  df_churn_trend = run_query(churn_query)
  df_cac_trend = run_query(cac_ltv_query)

  return {
      'params': {'market': market_list, 'start_date': start_date, 'end_date': end_date},
      'schema': schema,
      'mrr_data': df_mrr_trend.to_dict(orient='records'),
      'churn_data': df_churn_trend.to_dict(orient='records'),
      'cac_data': df_cac_trend.to_dict(orient='records')
  }


orchas_result = orchastrator(market=['All'], start_date = start_date, end_date = end_date)

#Overview agent

In [9]:
def agent_overview(context: dict) -> dict:

  schema = json.dumps(json.loads(context['schema'])['tables'])

  #########
  #MRR data
  #########
  mrr_data = pd.DataFrame(context['mrr_data'])
  total_mrr = mrr_data['monthly_revenue'].sum()
  top_mrr_market = mrr_data.groupby('market')['monthly_revenue'].sum().idxmax()
  top_mrr_segment = mrr_data.groupby('segment')['monthly_revenue'].sum().idxmax()
  top_mrr_plan = mrr_data.groupby('plan')['monthly_revenue'].sum().idxmax()
  top_mrr_product = mrr_data.groupby('product')['monthly_revenue'].sum().idxmax()

  ##MRR breakdown
  mkt_pct = (mrr_data.groupby('market')['monthly_revenue'].sum() / total_mrr * 100).round(1).to_dict()
  segment_pct = (mrr_data.groupby('segment')['monthly_revenue'].sum() / total_mrr * 100).round(1).to_dict()
  plan_pct = (mrr_data.groupby('plan')['monthly_revenue'].sum() / total_mrr * 100).round(1).to_dict()
  product_pct = (mrr_data.groupby('product')['monthly_revenue'].sum() / total_mrr * 100).round(1).to_dict()

  mrr_by_month = mrr_data.groupby('month')['monthly_revenue'].sum()
  first_month = mrr_by_month.iloc[0]
  last_month = mrr_by_month.iloc[-1]
  mrr_growth = ((last_month - first_month) / first_month * 100).round(1)

  #########
  #Churn data
  active_customer = df_revenue[
      (df_revenue['month'] >= context['params']['start_date'][:7]) &
       (df_revenue['month'] <= context['params']['end_date'][:7]) &
      (df_revenue['event_type'] != 'churn')
       ]['customer_id'].nunique()

  churn_data = pd.DataFrame(context['churn_data'])
  total_churn_customer = churn_data['customers_lost'].sum()

  #Churn breakdown
  top_churn_reasons = churn_data.groupby('churn_reason_category')['customers_lost'].sum().idxmax()
  top_churn_market = churn_data.groupby('market')['customers_lost'].sum().idxmax()
  top_churn_segment = churn_data.groupby('segment')['customers_lost'].sum().idxmax()
  top_churn_plan = churn_data.groupby('plan')['customers_lost'].sum().idxmax()

  #########
  #CAC Data
  #########
  cac_data = pd.DataFrame(context['cac_data'])
  avg_cac      = cac_data['avg_cac'].mean().round(2)
  avg_ltv      = cac_data['avg_ltv'].mean().round(2)
  avg_ltv_cac  = cac_data['ltv_cac_ratio'].mean().round(2)
  top_cac_channel = cac_data.groupby('acquisition_channel')['avg_cac'].mean().idxmax()


  prompt=f"""
  You are a data analyst at an accounting SaaS company.

  You are goal is to analyze the data and provide overview insights to the business.

  ### Data Dictionary
  {schema}

  ### Paramaters
  Markets: {context['params']['market']} | Period: {context['params']['start_date']} to {context['params']['end_date']}

  ###MRR Data
  {mrr_data.to_string(index=False)}

  ###Churn Data
  {churn_data.to_string(index=False)}

  ## Pre-calculated Numbers (use these exactly)
  - Total MRR: ${total_mrr:,.0f}
  - MRR Growth: {mrr_growth}%
  - Top Market: {top_mrr_market}
  - Top Segment: {top_mrr_segment}
  - Top Plan: {top_mrr_plan}
  - Top Product: {top_mrr_product}
  - Market MRR share: {mkt_pct}
  - Segment MRR share: {segment_pct}
  - Plan MRR share: {plan_pct}
  - Product MRR share: {product_pct}
  - Total Churned Customers: {total_churn_customer}
  - Total Active Customers: {active_customer}
  - Top Churn Reason: {top_churn_reasons}
  - Top Churn Market: {top_churn_market}
  - Top Churn Segment: {top_churn_segment}
  - Top Churn Plan: {top_churn_plan}
  - Avg CAC: ${avg_cac:,.0f}
  - Avg LTV: ${avg_ltv:,.0f}
  - LTV:CAC Ratio: {avg_ltv_cac}x
  - Most Expensive Channel: {top_cac_channel}

  ## Instructions
  Analyze the data above and write a structured overview with exactly these 4 sections,
  separated by a divider line.

  - You MUST reference specific numbers from the data in every sentence
  - Do NOT make general statements without backing them up with exact figures
  - Refer to the pre-calculated numbers for insight generation
  - Format numbers clearly: $1,234 for revenue, 12.3% for percentages
  - Note: Payroll MRR scales with headcount, focus on Accounting and Compliance for product mix insight

  ────────────────────────────────────────────────────
  Overall Performance:
  [2 sentences on overall MRR trend and growth]

  ────────────────────────────────────────────────────
  Key Highlights:
  [3 sentences on notable trends across market, segment, or plan]

  ────────────────────────────────────────────────────
  Revenue Mix:
  [2 sentences on which market/segment/plan is driving most revenue]

  ────────────────────────────────────────────────────
  Customer Churn:
  [2 sentences on which market/segment/plan is driving most churn]

  ────────────────────────────────────────────────────
  Customer Acquisition Cost:
  [2 sentences on which channel is driving most CAC]

  ────────────────────────────────────────────────────
  Watch Out:
  [1 sentence on the most concerning trend in the data]
  """

  return model.generate_content(
      prompt,
      generation_config=genai.GenerationConfig(temperature=0.3)
      ).text

overview_result = agent_overview(orchas_result)

print(overview_result)

Overall Performance:
The total MRR stands at $20,812,437, demonstrating a significant MRR growth of 1300.7%.

────────────────────────────────────────────────────
Key Highlights:
Singapore (SG) leads in MRR share at 50.0%, followed by Hong Kong (HK) at 21.4%, indicating a strong performance in these Asian markets. The Corporate segment contributes 41.7% to MRR, highlighting its importance, while the Full Compliance plan accounts for 48.2% of MRR.

────────────────────────────────────────────────────
Revenue Mix:
Singapore (SG) is the top market for revenue, contributing 50.0% of the total MRR. The Corporate segment drives 41.7% of the total MRR, and the Payroll product generates the highest MRR share at 69.1%.

────────────────────────────────────────────────────
Customer Churn:
Singapore (SG) has the highest churn rate, with Startups being the most affected segment and the Starter plan being the most churned plan. The data shows a total of 649 churned customers, with Price being the m

#Anomaly Detector Agent

In [10]:
def agent_anomaly_detector(context: dict,
                            churn_threshold=50,
                            mrr_neg_months=2,
                            ltv_cac_threshold=3,
                            revenue_mix_threshold=0.7,
                            cac_threshold=2) -> dict:

  mrr_data = pd.DataFrame(context['mrr_data'])
  churn_data = pd.DataFrame(context['churn_data'])

  #MRR monthly trend
  mrr_by_month = (
      mrr_data.groupby('month')['monthly_revenue'].sum()
      .reset_index()
      .sort_values('month')
  )

  mrr_by_month['mrr_growth'] = mrr_by_month['monthly_revenue'].pct_change() * 100

  #Churn
  churn_by_month = (
      churn_data.groupby('month')['customers_lost'].sum()
      .reset_index().rename(columns={'customers_lost': "total_churn"})
      .sort_values('month')
  )

  churn_by_month['3m_average'] = churn_by_month['total_churn'].rolling(3, min_periods=1).mean().shift(1)
  churn_by_month['churn_vs_baseline'] = (
      (churn_by_month['total_churn'] - churn_by_month['3m_average'])
      / churn_by_month['3m_average'] * 100
  ).round(1)


  #Churn by market
  churn_by_market = (
      churn_data.groupby(['month', 'market'])['customers_lost'].sum()
      .reset_index()
  )

  # Revenue mix
  total_mrr = mrr_data['monthly_revenue'].sum()
  market_pct = (mrr_data.groupby('market')['monthly_revenue'].sum() / total_mrr).to_dict()
  segment_pct = (mrr_data.groupby('segment')['monthly_revenue'].sum() / total_mrr).to_dict()
  plan_pct = (mrr_data.groupby('plan')['monthly_revenue'].sum() / total_mrr).to_dict()
  product_pct = (mrr_data.groupby('product')['monthly_revenue'].sum() / total_mrr).to_dict()

  prompt=f"""
  You are a data analyst at an accounting SaaS company.

  You are goal is to analyze the data and report any anomalies you observed based on the trends.

  Market: {context['params']['market']} | Period: {context['params']['start_date']} to {context['params']['end_date']}

  #MRR Trend (monthly, all markets)
  {mrr_by_month.to_string(index=False)}

  #Churn Trend(monthly, all markets)
  {churn_by_month.to_string(index=False)}

  #Churn by Market (monthly)
  {churn_by_market.to_string(index=False)}

  #Revenue Mix
  Market share: {market_pct}
  Segment share: {segment_pct}
  Plan share: {plan_pct}
  Product share: {product_pct}

  #Anomaly rules
  - MRR: flag out anomaly if the MRR trend has been negative for {mrr_neg_months} consecutive months
  - Churn: flag out anomaly if churn_vs_baseline is > +{churn_threshold}%. Use exact value from [churn_by_month] table only
  - LTV:CAC < {ltv_cac_threshold}x → flag. LTV:CAC > {ltv_cac_threshold}x is healthy, do NOT flag
  - Revenue Mix: flag ONLY if one market/segment/plan > {revenue_mix_threshold * 100}%. Exclude Payroll from product check
  - CAC: not applicable without customer-level data

  ## Drill Down Rules:
  - For churn spikes: cross-reference churn by market table, set drill_down_market to market with highest customers_lost that month
  - For market concentration: set drill_down_market to that market
  - If not market-specific: set drill_down_market to null

  ## Critical Instructions:
  - Only apply rules listed above, do not invent new rules
  - Do NOT flag if threshold is not strictly exceeded
  - Do NOT include anomaly if your observation says threshold is not met
  - severity must be "High" or "Medium" only

  Return ONLY a JSON object, no explanation, no markdown fences:
  {{
    "has_anomaly": true or false,
    "anomalies": [
      {{
        "metric": "MRR | Churn | LTV:CAC | Revenue Mix",
        "month": "YYYY-MM or null",
        "observation": "one sentence with exact numbers from data",
        "severity": "High | Medium",
        "possible_cause": "one sentence hypothesis",
        "drill_down_market": "SG | HK | AU | UK | null"
      }}
    ],
    "summary": "one sentence summary"
  }}
  """
  raw    = model.generate_content(prompt, generation_config=genai.GenerationConfig(temperature=0.0)).text
  clean  = raw.replace("```json", "").replace("```", "").strip()
  result = json.loads(clean)

  # ── Python filter ─────────────────────────────────────────────────────────
  churn_lookup = churn_by_month.set_index('month')['churn_vs_baseline'].to_dict()
  neg_growth   = mrr_by_month['mrr_growth'] < 0
  consecutive_neg = neg_growth & neg_growth.shift(1, fill_value=False)

  filtered = []
  for a in result['anomalies']:
      if a['metric'] == 'Churn':
          if abs(churn_lookup.get(a['month'], 0)) <= churn_threshold:
              continue
      elif a['metric'] == 'Revenue Mix':
          all_pcts = {**market_pct, **segment_pct, **plan_pct}
          if not any(v > revenue_mix_threshold for v in all_pcts.values()):
              continue
      elif a['metric'] == 'MRR':
          if not consecutive_neg.any():
              continue
      filtered.append(a)

  result['anomalies'] = filtered
  result['has_anomaly'] = len(filtered) > 0

  # ── Print ──────────────────────────────────────────────────────────────────
  print("=" * 55)
  print("🔍 ANOMALY DETECTOR")
  print("=" * 55)
  print(f"\n{result['summary']}\n")

  if result['has_anomaly']:
      print(f"⚠️  {len(result['anomalies'])} anomaly/anomalies detected:\n")
      for a in result['anomalies']:
          month_str  = f"[{a['month']}] " if a['month'] else ""
          market_str = f"  → Drill down: {a['drill_down_market']}" if a.get('drill_down_market') else ""
          print(f"  {a['metric']} {month_str}[{a['severity']}]")
          print(f"  → {a['observation']}")
          print(f"  → Possible cause: {a['possible_cause']}")
          if market_str:
              print(market_str)
          print()
  else:
      print("✅ No anomalies detected — all metrics within normal range")

  print("=" * 55)
  return result

In [11]:
anomaly = agent_anomaly_detector(orchas_result)

🔍 ANOMALY DETECTOR

Anomalies were detected in churn rates, with significant spikes in April 2023, September 2023, and July 2024, exceeding the +50% threshold, and a moderate concern regarding revenue concentration in Singapore.

⚠️  3 anomaly/anomalies detected:

  Churn [2023-04] [High]
  → Churn vs baseline was 75.6%, exceeding the +50% threshold.
  → Possible cause: A significant increase in customer churn occurred in April 2023, potentially due to a product issue or competitive pressure.
  → Drill down: SG

  Churn [2023-09] [High]
  → Churn vs baseline was 76.8%, exceeding the +50% threshold.
  → Possible cause: Another substantial churn event happened in September 2023, indicating a recurring issue or a new factor impacting customer retention.
  → Drill down: SG

  Churn [2024-07] [High]
  → Churn vs baseline was 65.5%, exceeding the +50% threshold.
  → Possible cause: A significant churn spike occurred in July 2024, indicating a new or resurfaced issue impacting customer retent

#Deepdive Agent

In [26]:
def agent_deep_dive(anomaly_result: dict, start_date: str, end_date: str) -> dict:

    flagged_markets = set(
        a['drill_down_market']
        for a in anomaly_result['anomalies']
        if a.get('drill_down_market') and a['drill_down_market'] not in [None, 'null', '']
    )

    print('flagged markets: ', flagged_markets)

    if not flagged_markets:
        print("No market-specific anomalies to drill down.")
        return {}

    print('Flagged_markets: ', flagged_markets)

    deep_dive_results = {}

    for market in flagged_markets:

        print('Market: ', market)

        # Re-run orchestrator for specific market
        mkt_ctx = orchastrator(market=[market], start_date=start_date, end_date=end_date)

        # Get anomalies for this market
        mkt_anomalies = [
            a for a in anomaly_result['anomalies']
            if a.get('drill_down_market') == market
        ]

        # Pre-calculate from raw data
        mrr_df   = pd.DataFrame(mkt_ctx['mrr_data'])
        churn_df = pd.DataFrame(mkt_ctx['churn_data'])
        cac_df   = pd.DataFrame(mkt_ctx['cac_data'])

        # MRR breakdown
        total_mrr    = mrr_df['monthly_revenue'].sum()
        segment_pct  = (mrr_df.groupby('segment')['monthly_revenue'].sum() / total_mrr * 100).round(1).to_dict()
        plan_pct     = (mrr_df.groupby('plan')['monthly_revenue'].sum() / total_mrr * 100).round(1).to_dict()

        # Churn breakdown
        churn_by_segment = churn_df.groupby('segment')['customers_lost'].sum().to_dict()
        churn_by_plan    = churn_df.groupby('plan')['customers_lost'].sum().to_dict()
        churn_by_reason  = churn_df.groupby('churn_reason_category')['customers_lost'].sum().to_dict()

        # CAC / LTV
        avg_cac     = cac_df['avg_cac'].mean().round(2)
        avg_ltv_cac = cac_df['ltv_cac_ratio'].mean().round(2)
        top_cac_channel = cac_df.groupby('acquisition_channel')['avg_cac'].mean().idxmax()

        prompt = f"""
        You are a senior sales analyst at Sleek, an accounting SaaS company.
        You are deep diving into the {market} market to identify root causes of detected anomalies.

        ## Detected Anomalies in {market}
        {json.dumps(mkt_anomalies, indent=2)}

        ## {market} Pre-calculated Metrics
        - Total MRR: ${total_mrr:,.0f}
        - MRR by Segment (%): {segment_pct}
        - MRR by Plan (%): {plan_pct}
        - Churn by Segment: {churn_by_segment}
        - Churn by Plan: {churn_by_plan}
        - Churn by Reason: {churn_by_reason}
        - Avg CAC: ${avg_cac:,.0f}
        - LTV:CAC Ratio: {avg_ltv_cac}x
        - Most Expensive Channel: {top_cac_channel}

        ## {market} MRR Trend
        {mrr_df.groupby('month')['monthly_revenue'].sum().reset_index().to_string(index=False)}

        ## {market} Churn Trend
        {churn_df.groupby('month')['customers_lost'].sum().reset_index().to_string(index=False)}

        ## Instructions
        - Focus on identifying WHICH segment / plan / channel is driving the anomaly
        - Cross-reference MRR and churn breakdowns to find the most likely driver
        - Be specific with numbers from the pre-calculated metrics

        Return ONLY a JSON object, no explanation, no markdown fences:
        {{
          "market": "{market}",
          "anomaly_summary": "one sentence describing the anomaly in this market with exact numbers",
          "drivers": [
            {{
              "dimension": "Segment | Plan | Channel | Product",
              "driver": "specific value e.g. Startup, Paid, Starter",
              "observation": "one sentence with exact numbers",
              "contribution": "High | Medium"
            }}
          ],
          "root_cause": "2 sentences synthesizing the most likely root cause with numbers",
          "recommended_action": "one sentence on what to investigate or action to take"
        }}
        """

        raw    = model.generate_content(prompt, generation_config=genai.GenerationConfig(temperature=0.1)).text
        clean  = raw.replace("```json", "").replace("```", "").strip()
        result = json.loads(clean)
        deep_dive_results[market] = result

        # Print
        print("=" * 55)
        print(f"🔬 DEEP DIVE — {market}")
        print("=" * 55)
        print(f"\n{result['anomaly_summary']}\n")
        print("── Drivers ──────────────────────────────────────────")
        for d in result['drivers']:
            print(f"  [{d['dimension']}] {d['driver']} ({d['contribution']})")
            print(f"  → {d['observation']}\n")
        print("── Root Cause ───────────────────────────────────────")
        print(f"  {result['root_cause']}\n")
        print("── Recommended Action ───────────────────────────────")
        print(f"  {result['recommended_action']}")
        print("=" * 55)

    return deep_dive_results

#Summary Agent

In [27]:
def agent_summary(context: dict, overview: str, anomaly_result: dict, deep_dive: dict) -> str:

    # Prepare deep dive text
    deep_dive_str = ""
    if deep_dive:
        for market, result in deep_dive.items():
            deep_dive_str += f"\n### {market}\n"
            deep_dive_str += f"Anomaly: {result.get('anomaly_summary', '')}\n"
            deep_dive_str += f"Root Cause: {result.get('root_cause', '')}\n"
            deep_dive_str += f"Recommended Action: {result.get('recommended_action', '')}\n"
    else:
        deep_dive_str = "No market-specific anomalies detected."

    prompt = f"""
    You are a senior sales analyst at an accounting SaaS company.
    Write a concise executive summary for the leadership team.

    ## Parameters
    Market: {context['params']['market']} | Period: {context['params']['start_date']} to {context['params']['end_date']}

    ## Overview Analysis
    {overview}

    ## Anomaly Detection Summary
    {anomaly['summary']}

    ## Anomalies Found
    {json.dumps(anomaly['anomalies'], indent=2) if anomaly['has_anomaly'] else "No anomalies detected."}

    ## Deep Dive Findings
    {deep_dive_str}

    ## Instructions
    - Write for a leadership audience — clear, concise, actionable
    - Must include specific numbers in every section
    - No bullet points, write in prose

    Write exactly these 3 sections separated by a divider:

    ────────────────────────────────────────────────────
    What Happened:
    [2-3 sentences — overall performance + key anomalies found]

    ────────────────────────────────────────────────────
    Why It Happened:
    [2-3 sentences — root causes from deep dive, specific market/segment/plan]

    ────────────────────────────────────────────────────
    What To Do:
    [3 prioritized actions, numbered 1-3, one sentence each]
    """

    result = model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(temperature=0.4)
    ).text

    print("=" * 55)
    print("📝 EXECUTIVE SUMMARY")
    print("=" * 55)
    print(result)
    print("=" * 55)

    return result

#Pipeline

In [28]:
def run_pipeline(market = ['All'], start_date = start_date, end_date = end_date):

    print("\n" + "=" * 55)
    print("🚀 PIPELINE STARTED")
    print(f"   Market: {market} | {start_date} to {end_date}")
    print("=" * 55 + "\n")

    #Step 1: Orchastrator
    print("⚙️  Step 1: Orchestrating data...\n")
    ctx = orchastrator(market=market, start_date=start_date, end_date=end_date)

    #Step 2: Overview
    print("⚙️  Step 2: Running Overview Agent...\n")
    overview = agent_overview(ctx)

    #Step 3: Anomaly detector
    print("⚙️  Step 3: Running Anomaly Detector...\n")
    anomaly = agent_anomaly_detector(ctx)

    #Step 4: Deep dive, only if anomaly is found
    deep_dive = {}

    if anomaly['has_anomaly']:
      print("⚙️  Step 4: Anomaly found, running Anomaly Detector...\n")
      deepdive = agent_deep_dive(anomaly, start_date=start_date, end_date=end_date)

    else:
      print("⚙️  Step 4: No anomalies — Skipping Deep Dive\n")

    #Step 5: Summary
    print("⚙️  Step 5: Running Summary Agent...\n")
    summary = agent_summary(context = ctx, overview = overview, deep_dive = deepdive, anomaly_result = anomaly)

    print("\n" + "=" * 55)
    print("✅ PIPELINE COMPLETE")
    print("=" * 55 + "\n")

    return {
        'context':   ctx,
        'overview':  overview,
        'anomaly':   anomaly,
        'deep_dive': deep_dive,
        'summary':   summary,
    }

#Generate demo data

In [15]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/Gen-AI/sales-agent/result'
os.makedirs(save_dir, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
scenarios = [
    {'market': ['All'], 'start_date': '2022-01-01', 'end_date': '2024-12-31'},
    {'market': ['HK'],  'start_date': '2022-01-01', 'end_date': '2024-12-31'},
    {'market': ['SG'],  'start_date': '2022-01-01', 'end_date': '2024-12-31'},
    {'market': ['AU'],  'start_date': '2022-01-01', 'end_date': '2024-12-31'},
    {'market': ['UK'],  'start_date': '2022-01-01', 'end_date': '2024-12-31'},
]

for s in scenarios:
    print(f"Running {s['market'][0]}...")


Running All...
Running HK...
Running SG...
Running AU...
Running UK...


In [30]:
import json
import time

scenarios = [
    {'market': ['All'], 'start_date': '2022-01-01', 'end_date': '2024-12-31'},
    {'market': ['HK'],  'start_date': '2022-01-01', 'end_date': '2024-12-31'},
    {'market': ['SG'],  'start_date': '2022-01-01', 'end_date': '2024-12-31'},
    {'market': ['AU'],  'start_date': '2022-01-01', 'end_date': '2024-12-31'},
    {'market': ['UK'],  'start_date': '2022-01-01', 'end_date': '2024-12-31'},
]

for s in scenarios:
    print(f"Running {s['market'][0]}...")
    result = run_pipeline(**s)

    filename = f"{save_dir}/{s['market'][0]}_2022_2024.json"
    with open(filename, 'w') as f:
        json.dump(result, f, indent=2, default=str)

    print(f"Saved {filename}\n")

    time.sleep(30)

Running All...

🚀 PIPELINE STARTED
   Market: ['All'] | 2022-01-01 to 2024-12-31

⚙️  Step 1: Orchestrating data...

⚙️  Step 2: Running Overview Agent...

⚙️  Step 3: Running Anomaly Detector...

🔍 ANOMALY DETECTOR

Anomalies were detected in churn rates, with several months exceeding the +50% threshold, particularly in Singapore, while revenue mix shows no single market exceeding 70% concentration.

⚠️  4 anomaly/anomalies detected:

  Churn [2022-04] [High]
  → Churn vs baseline was 566.7%, significantly exceeding the +50% threshold.
  → Possible cause: A sudden increase in customer dissatisfaction or a specific event impacting customers in Singapore.
  → Drill down: SG

  Churn [2023-04] [Medium]
  → Churn vs baseline was 75.6%, exceeding the +50% threshold.
  → Possible cause: A potential issue with customer retention in Singapore during this period.
  → Drill down: SG

  Churn [2023-09] [Medium]
  → Churn vs baseline was 76.8%, exceeding the +50% threshold.
  → Possible cause: A 

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 992.21ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2605.95ms


📝 EXECUTIVE SUMMARY
────────────────────────────────────────────────────
What Happened:
Our accounting SaaS has achieved exceptional MRR growth of 938.9% to $10,407,948 within the 2022-2024 period, with Singapore (SG) representing 100% of this revenue. However, we experienced significant churn anomalies in April 2023 (75.6% above baseline), September 2023 (76.8% above baseline), and July 2024 (65.5% above baseline), indicating a critical retention challenge.

────────────────────────────────────────────────────
Why It Happened:
The high churn rates, particularly in SG, are primarily driven by issues within the 'Starter' plan and the 'Startup' customer segment, with 'Product' and 'Price' cited as the main reasons. This over-reliance on a single market (SG) magnifies the impact of these segment-specific and product-related churn drivers, creating substantial risk.

────────────────────────────────────────────────────
What To Do:
1. Conduct an immediate deep-dive analysis into customer fe